In [5]:
!pip install -q -U pinecone sentence-transformers tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 41.1 MB/s eta 0:00:00


In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device name: Tesla T4


In [3]:
from getpass import getpass
PINECONE_API_KEY = getpass("Enter your Pinecone API key: ")

Enter your Pinecone API key: ··········


In [7]:
# ==============================================================
# CELL 1 — Install dependencies (run once per Colab session)
# ==============================================================
# !pip install -q -U pinecone sentence-transformers tqdm

# ==============================================================
# CELL 2 — Check GPU (make sure Runtime > Change runtime type > T4 GPU)
# ==============================================================
# import torch
# print("GPU available:", torch.cuda.is_available())
# print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ==============================================================
# CELL 3 — Get your Pinecone API key (no .env file needed in Colab)
# ==============================================================
# from getpass import getpass
# PINECONE_API_KEY = getpass("Enter your Pinecone API key: ")

# ==============================================================
# CELL 4 — Upload your CSV (or mount Drive if it's stored there)
# ==============================================================
# Option A: direct upload
# from google.colab import files
# uploaded = files.upload()  # select processed_medmcqa.csv from your machine
#
# Option B: mount Google Drive instead (uncomment if your CSV lives in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# csv_path = "/content/drive/MyDrive/path/to/processed_medmcqa.csv"

# ==============================================================
# CELL 5 — Main ingestion script
# ==============================================================
import time
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm  # notebook-friendly progress bar for Colab
from concurrent.futures import ThreadPoolExecutor

from pinecone import Pinecone, ServerlessSpec

# --- Pinecone init ---
# Use PINECONE_API_KEY from Cell 3 (getpass) instead of load_dotenv/.env
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "cdf-rag-index"

if index_name not in pc.list_indexes().names():
    print(f"Creating Pinecone index: {index_name}...")
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    time.sleep(5)

index = pc.Index(index_name)

# --- Load model on GPU, fp16 for T4 speed ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading embedding model...")
model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)
if device == "cuda":
    model = model.half()
print("Embedding model loaded.")

# --- Config ---
EMBED_BATCH = 256          # rows encoded per GPU forward pass
UPSERT_CHUNK = 2000        # vectors accumulated before triggering a background upload
UPSERT_BATCH_SIZE = 200    # internal batch size the SDK splits each upload into
MAX_CONCURRENCY = 10       # internal thread pool size the SDK uses per upload call
MAX_PENDING_UPLOADS = 2    # how many upload chunks may be in flight before we block

# Background executor so upserts run while the GPU keeps embedding the next chunk.
# The SDK's own upsert() already parallelizes internally via batch_size/max_concurrency,
# so we only need a small pool here to overlap chunk-level uploads with embedding.
upload_executor = ThreadPoolExecutor(max_workers=MAX_PENDING_UPLOADS)
pending_uploads = []


def submit_upload(vectors_chunk):
    # Cap in-flight uploads so we don't buffer unbounded chunks in memory
    while len(pending_uploads) >= MAX_PENDING_UPLOADS:
        pending_uploads.pop(0).result()
    fut = upload_executor.submit(
        index.upsert,
        vectors=vectors_chunk,
        batch_size=UPSERT_BATCH_SIZE,
        max_concurrency=MAX_CONCURRENCY,
    )
    pending_uploads.append(fut)


# --- Load data ---
csv_path = "processed_medmcqa.csv"  # change if using the Drive path from Cell 4
df = pd.read_csv(csv_path).fillna("")
print(f"Loaded {len(df)} documents.")
print("🚀 Embedding + upserting to Pinecone (async pipeline)...")

texts_all = df["document"].astype(str).tolist()
vectors_buffer = []

pbar = tqdm(total=len(df), desc="Embedding+Upserting", unit="docs")

for start in range(0, len(df), EMBED_BATCH):
    end = min(start + EMBED_BATCH, len(df))
    batch_rows = df.iloc[start:end]
    batch_texts = texts_all[start:end]

    with torch.inference_mode():
        embeddings = model.encode(
            batch_texts,
            batch_size=EMBED_BATCH,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

    for row, vector in zip(batch_rows.itertuples(index=False), embeddings):
        vectors_buffer.append({
            "id": f"question_{row.id}",
            "values": vector.tolist(),
            "metadata": {
                "text": row.document,
                "question": row.question,
                "answer": row.answer,
                "subject": row.subject,
                "topic": row.topic,
            },
        })

    if len(vectors_buffer) >= UPSERT_CHUNK:
        submit_upload(vectors_buffer.copy())
        vectors_buffer.clear()

    pbar.update(len(batch_rows))

if vectors_buffer:
    submit_upload(vectors_buffer)

# wait for any remaining background uploads to finish
for fut in pending_uploads:
    fut.result()
upload_executor.shutdown(wait=True)
pbar.close()

print("\n✅ Pinecone ingestion completed.")

Using device: cuda
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Loaded 182822 documents.
🚀 Embedding + upserting to Pinecone (async pipeline)...


Embedding+Upserting:   0%|          | 0/182822 [00:00<?, ?docs/s]

Upserting:   0%|          | 0/11 [00:00<?, ?batch/s]

Upserting:   0%|          | 0/11 [00:00<?, ?batch/s]

... (85 outputs left out)

Upserting:   0%|          | 0/11 [00:00<?, ?batch/s]

Upserting:   0%|          | 0/11 [00:00<?, ?batch/s]

Upserting:   0%|          | 0/3 [00:00<?, ?batch/s]


✅ Pinecone ingestion completed.
